In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("ecommerce.db")

def run_query(sql):
    return pd.read_sql_query(sql, conn)

In [2]:
# Query 1: Total revenue per category
q1 = """
SELECT 
    p.category,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""
run_query(q1)

,category,total_revenue
0,Electronics,4195817.10
1,Books,3947225.83
2,Clothing,3693461.05
3,Home,3508774.43


In [3]:
# Query 2: Top 10 customers by total order value
q2 = """
SELECT 
    o.customer_id,
    c.customer_name,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)), 2) AS total_order_value
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id, c.customer_name
ORDER BY total_order_value DESC
LIMIT 10;
"""
run_query(q2)

,customer_id,customer_name,total_order_value
0,306,Patricia Lee,198393.35
1,41,Jennifer Good,180004.80
2,287,Scott Burton,141972.61
3,383,Christopher Reynolds,139595.70
4,316,Anthony Williams,138630.01
5,303,Sharon Bonilla,137027.56
6,36,Marcus Taylor,133687.79
7,471,Aaron Parker,132564.65
8,406,Elizabeth Mcdaniel,132168.90
9,223,Jordan Mahoney,131832.38


In [4]:
# Query 3: Month-wise order count for the last 12 months
q3 = """
SELECT 
    strftime('%Y-%m', order_date) AS order_month,
    COUNT(DISTINCT order_id) AS order_count
FROM orders
WHERE order_date >= date('now', '-12 months')
GROUP BY order_month
ORDER BY order_month;
"""
run_query(q3)

,order_month,order_count
0,2025-08,59
1,2025-09,63
2,2025-10,73
3,2025-11,67
4,2025-12,70
5,2026-01,66
6,2026-02,70
7,2026-03,54
8,2026-04,52
9,2026-05,81


In [5]:
# Query 4: Customers who placed orders but never had any item delivered
q4 = """
SELECT DISTINCT c.customer_id, c.customer_name
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
WHERE c.customer_id NOT IN (
    SELECT customer_id FROM orders WHERE status = 'DELIVERED' AND customer_id IS NOT NULL
);
"""
run_query(q4)

,customer_id,customer_name
0,1,Alexander Meyers
1,3,Tyler Wilson
2,5,Alexis Dawson
3,9,Mr. Earl Campbell
4,13,Adrian Gonzalez
...,...,...
244,492,Patrick Ortiz
245,493,Christina Smith
246,495,Susan Thornton
247,497,Mackenzie Collins


In [6]:
# Query 5: Products that were ordered but had more returns than purchases
q5 = """
SELECT 
    p.product_id,
    p.product_name,
    SUM(CASE WHEN oi.quantity > 0 THEN oi.quantity ELSE 0 END) AS purchased_qty,
    SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS returned_qty
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.product_id, p.product_name
HAVING returned_qty > purchased_qty;
"""
run_query(q5)

,product_id,product_name,purchased_qty,returned_qty
0,1,Member Comics,2,4
1,6,Whatever Mobiles,2,10
2,23,Section Non-Fiction,0,10
3,109,Lot Bedding,9,10
4,112,Listen Kitchen,8,9
5,438,Generation Women,0,8
6,449,Office Mobiles,7,8


In [7]:
# Query 6: Return rate (returned items / total items) per category
q6 = """
SELECT 
    p.category,
    SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) AS returned_qty,
    SUM(ABS(oi.quantity)) AS total_qty,
    ROUND(
        SUM(CASE WHEN oi.quantity < 0 THEN -oi.quantity ELSE 0 END) * 100.0 / SUM(ABS(oi.quantity)), 
        2
    ) AS return_rate_percent
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY return_rate_percent DESC;
"""
run_query(q6)

,category,returned_qty,total_qty,return_rate_percent
0,Home,102,2617,3.90
1,Clothing,88,2696,3.26
2,Books,68,2707,2.51
3,Electronics,71,2836,2.50


In [8]:
# Query 7: Running total of revenue per region, ordered by date
q7 = """
WITH daily_rev AS (
    SELECT 
        o.region_code,
        DATE(o.order_date) AS order_date,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS daily_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.region_code, DATE(o.order_date)
)
SELECT 
    region_code,
    order_date,
    ROUND(daily_revenue, 2) AS daily_revenue,
    ROUND(SUM(daily_revenue) OVER (
        PARTITION BY region_code 
        ORDER BY order_date
    ), 2) AS running_total
FROM daily_rev
ORDER BY region_code, order_date;
"""
run_query(q7)

,region_code,order_date,daily_revenue,running_total
0,EAST,2025-08-05,19918.55,19918.55
1,EAST,2025-08-07,48316.04,68234.59
2,EAST,2025-08-13,49253.24,117487.83
3,EAST,2025-08-15,34258.12,151745.95
4,EAST,2025-08-16,23496.56,175242.51
...,...,...,...,...
577,WEST,2026-07-28,2821.09,3408557.96
578,WEST,2026-07-29,17277.26,3425835.22
579,WEST,2026-07-30,989.54,3426824.76
580,WEST,2026-07-31,3597.32,3430422.09


In [9]:
# Query 8: Rank products by total revenue within each category (DENSE_RANK)
q8 = """
WITH product_rev AS (
    SELECT 
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category, p.product_name
)
SELECT 
    category,
    product_name,
    ROUND(total_revenue, 2) AS total_revenue,
    DENSE_RANK() OVER (
        PARTITION BY category 
        ORDER BY total_revenue DESC
    ) AS rank_in_category
FROM product_rev
ORDER BY category, rank_in_category;
"""
run_query(q8)

,category,product_name,total_revenue,rank_in_category
0,Books,View Academic,105803.08,1
1,Books,Reach Comics,96528.65,2
2,Books,Trip Fiction,89037.75,3
3,Books,Like Academic,82622.63,4
4,Books,Until Comics,77834.06,5
...,...,...,...,...
479,Home,Customer Kitchen,1800.41,108
480,Home,Lot Furniture,1496.42,109
481,Home,Respond Decor,568.60,110
482,Home,Commercial Decor,203.10,111


In [10]:
# Query 9: Days between consecutive orders per customer (LAG), flag "At Risk"
q9 = """
WITH customer_orders AS (
    SELECT 
        customer_id,
        order_date,
        LAG(order_date) OVER (
            PARTITION BY customer_id 
            ORDER BY order_date
        ) AS previous_order_date
    FROM orders
    WHERE customer_id IS NOT NULL AND customer_id != ''
),
gaps AS (
    SELECT 
        customer_id,
        order_date,
        previous_order_date,
        CASE 
            WHEN previous_order_date IS NOT NULL 
            THEN JULIANDAY(order_date) - JULIANDAY(previous_order_date)
            ELSE NULL 
        END AS days_gap
    FROM customer_orders
),
avg_gaps AS (
    SELECT customer_id, AVG(days_gap) AS avg_gap
    FROM gaps
    WHERE days_gap IS NOT NULL
    GROUP BY customer_id
)
SELECT 
    g.customer_id,
    g.order_date,
    g.previous_order_date,
    ROUND(g.days_gap, 1) AS days_gap,
    CASE WHEN a.avg_gap > 30 THEN 'At Risk' ELSE 'Active' END AS customer_status
FROM gaps g
JOIN avg_gaps a ON g.customer_id = a.customer_id
ORDER BY g.customer_id, g.order_date;
"""
run_query(q9)

,customer_id,order_date,previous_order_date,days_gap,customer_status
0,7,2026-02-23 12:19:55,NaN,NaN,At Risk
1,7,2026-04-21 10:08:23,2026-02-23 12:19:55,56.9,At Risk
2,7,2026-07-01 19:17:52,2026-04-21 10:08:23,71.4,At Risk
3,9,2026-01-08 04:31:46,NaN,NaN,At Risk
4,9,2026-03-13 23:36:19,2026-01-08 04:31:46,64.8,At Risk
...,...,...,...,...,...
592,496,2026-02-20 14:59:41,NaN,NaN,At Risk
593,496,2026-06-15 13:36:27,2026-02-20 14:59:41,114.9,At Risk
594,497,2026-04-27 11:00:45,NaN,NaN,Active
595,497,2026-05-18 03:52:21,2026-04-27 11:00:45,20.7,Active


In [11]:
# Query 10: CTE with Multiple Levels - customer categorization by monthly revenue
q10 = """
WITH monthly_customer_revenue AS (
    SELECT 
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS order_month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS monthly_revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL AND o.customer_id != ''
    GROUP BY o.customer_id, order_month
),
categorized AS (
    SELECT 
        customer_id,
        order_month,
        monthly_revenue,
        CASE 
            WHEN monthly_revenue > 10000 THEN 'High'
            WHEN monthly_revenue BETWEEN 5000 AND 10000 THEN 'Medium'
            ELSE 'Low'
        END AS revenue_category
    FROM monthly_customer_revenue
)
SELECT 
    order_month,
    revenue_category,
    COUNT(DISTINCT customer_id) AS customer_count
FROM categorized
GROUP BY order_month, revenue_category
ORDER BY order_month, revenue_category;
"""
run_query(q10)

,order_month,revenue_category,customer_count
0,2025-08,High,31
1,2025-08,Low,10
2,2025-08,Medium,9
3,2025-09,High,30
4,2025-09,Low,14
5,2025-09,Medium,6
6,2025-10,High,36
7,2025-10,Low,12
8,2025-10,Medium,8
9,2025-11,High,42


In [12]:
# Query 11: NTILE for Segmentation - quartiles based on lifetime value
q11 = """
WITH customer_ltv AS (
    SELECT 
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL AND o.customer_id != ''
    GROUP BY o.customer_id
)
SELECT 
    customer_id,
    ROUND(total_value, 2) AS total_value,
    NTILE(4) OVER (ORDER BY total_value DESC) AS quartile,
    CASE NTILE(4) OVER (ORDER BY total_value DESC)
        WHEN 1 THEN 'Platinum'
        WHEN 2 THEN 'Gold'
        WHEN 3 THEN 'Silver'
        WHEN 4 THEN 'Bronze'
    END AS quartile_label
FROM customer_ltv
ORDER BY total_value DESC;
"""
run_query(q11)

,customer_id,total_value,quartile,quartile_label
0,306,198393.35,1,Platinum
1,41,180004.80,1,Platinum
2,287,141972.61,1,Platinum
3,383,139595.70,1,Platinum
4,316,138630.01,1,Platinum
...,...,...,...,...
366,173,-14193.21,4,Bronze
367,208,-18084.35,4,Bronze
368,25,-21295.64,4,Bronze
369,429,-27701.54,4,Bronze


In [13]:
# Query 12: Year-over-Year Comparison
q12 = """
WITH monthly_revenue AS (
    SELECT 
        CAST(strftime('%Y', o.order_date) AS INTEGER) AS year,
        CAST(strftime('%m', o.order_date) AS INTEGER) AS month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY year, month
)
SELECT 
    curr.year,
    curr.month,
    ROUND(curr.revenue, 2) AS revenue,
    ROUND(prev.revenue, 2) AS prev_year_revenue,
    CASE 
        WHEN prev.revenue IS NOT NULL AND prev.revenue != 0
        THEN ROUND((curr.revenue - prev.revenue) * 100.0 / prev.revenue, 2)
        ELSE NULL
    END AS yoy_growth_percent
FROM monthly_revenue curr
LEFT JOIN monthly_revenue prev 
    ON curr.year = prev.year + 1 AND curr.month = prev.month
ORDER BY curr.year, curr.month;
"""
run_query(q12)

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2025,8,1336534.86,NaN,NaN
1,2025,9,1047197.98,NaN,NaN
2,2025,10,1256601.60,NaN,NaN
3,2025,11,1395164.84,NaN,NaN
4,2025,12,1344957.89,NaN,NaN
5,2026,1,1248624.52,NaN,NaN
6,2026,2,1055195.84,NaN,NaN
7,2026,3,836809.28,NaN,NaN
8,2026,4,1340209.53,NaN,NaN
9,2026,5,1738251.30,NaN,NaN


In [14]:
# Query 13: First/Last Value Analysis - first vs most recent purchased category
q13 = """
WITH customer_category_orders AS (
    SELECT 
        o.customer_id,
        o.order_date,
        p.category,
        FIRST_VALUE(p.category) OVER (
            PARTITION BY o.customer_id ORDER BY o.order_date ASC
        ) AS first_category,
        FIRST_VALUE(p.category) OVER (
            PARTITION BY o.customer_id ORDER BY o.order_date DESC
        ) AS last_category
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.customer_id IS NOT NULL AND o.customer_id != ''
)
SELECT DISTINCT
    customer_id,
    first_category,
    last_category,
    CASE WHEN first_category != last_category THEN 'Yes' ELSE 'No' END AS category_shift
FROM customer_category_orders
ORDER BY customer_id;
"""
run_query(q13)

,customer_id,first_category,last_category,category_shift
0,1,Electronics,Electronics,No
1,3,Home,Home,No
2,4,Electronics,Electronics,No
3,5,Books,Books,No
4,7,Electronics,Books,Yes
...,...,...,...,...
366,495,Home,Home,No
367,496,Home,Electronics,Yes
368,497,Books,Books,No
369,499,Home,Home,No


In [15]:
# Query 14: Cumulative Distribution - % of revenue from top N% of customers
q14 = """
WITH customer_revenue AS (
    SELECT 
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL AND o.customer_id != ''
    GROUP BY o.customer_id
),
ranked AS (
    SELECT 
        customer_id,
        revenue,
        SUM(revenue) OVER (ORDER BY revenue DESC) AS cumulative_revenue,
        SUM(revenue) OVER () AS total_revenue
    FROM customer_revenue
)
SELECT 
    customer_id,
    ROUND(revenue, 2) AS revenue,
    ROUND(cumulative_revenue, 2) AS cumulative_revenue,
    ROUND(cumulative_revenue * 100.0 / total_revenue, 2) AS cumulative_percent
FROM ranked
ORDER BY revenue DESC;
"""
run_query(q14)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,306,198393.35,198393.35,1.37
1,41,180004.80,378398.15,2.60
2,287,141972.61,520370.76,3.58
3,383,139595.70,659966.46,4.54
4,316,138630.01,798596.47,5.50
...,...,...,...,...
366,173,-14193.21,14629596.67,100.71
367,208,-18084.35,14611512.32,100.59
368,25,-21295.64,14590216.68,100.44
369,429,-27701.54,14562515.14,100.25


In [16]:
# Query 15: Cohort Analysis - retention by registration month
q15 = """
WITH cohorts AS (
    SELECT 
        customer_id,
        strftime('%Y-%m', registration_date) AS cohort_month
    FROM customers
),
customer_orders AS (
    SELECT 
        o.customer_id,
        strftime('%Y-%m', o.order_date) AS order_month
    FROM orders o
    WHERE o.customer_id IS NOT NULL AND o.customer_id != ''
),
cohort_activity AS (
    SELECT 
        c.customer_id,
        c.cohort_month,
        co.order_month,
        (CAST(strftime('%Y', co.order_month || '-01') AS INTEGER) * 12 + CAST(strftime('%m', co.order_month || '-01') AS INTEGER))
        -
        (CAST(strftime('%Y', c.cohort_month || '-01') AS INTEGER) * 12 + CAST(strftime('%m', c.cohort_month || '-01') AS INTEGER))
        AS month_number
    FROM cohorts c
    JOIN customer_orders co ON c.customer_id = co.customer_id
),
cohort_sizes AS (
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_size
    FROM cohorts
    GROUP BY cohort_month
)
SELECT 
    ca.cohort_month,
    ca.month_number,
    COUNT(DISTINCT ca.customer_id) AS customers_active,
    cs.cohort_size,
    ROUND(COUNT(DISTINCT ca.customer_id) * 100.0 / cs.cohort_size, 2) AS retention_rate_percent
FROM cohort_activity ca
JOIN cohort_sizes cs ON ca.cohort_month = cs.cohort_month
WHERE ca.month_number BETWEEN 0 AND 3
GROUP BY ca.cohort_month, ca.month_number
ORDER BY ca.cohort_month, ca.month_number;
"""
run_query(q15)

,cohort_month,month_number,customers_active,cohort_size,retention_rate_percent
0,2025-05,3,2,18,11.11
1,2025-06,2,2,21,9.52
2,2025-06,3,1,21,4.76
3,2025-07,1,1,23,4.35
4,2025-07,2,4,23,17.39
5,2025-07,3,3,23,13.04
6,2025-08,0,2,23,8.70
7,2025-08,1,2,23,8.70
8,2025-08,2,3,23,13.04
9,2025-09,0,1,15,6.67


In [17]:
# Query 16: Self-Join - products frequently bought together
q16 = """
SELECT 
    p1.product_name AS product_a,
    p2.product_name AS product_b,
    COUNT(*) AS times_bought_together
FROM order_items oi1
JOIN order_items oi2 
    ON oi1.order_id = oi2.order_id 
    AND oi1.product_id < oi2.product_id
JOIN products p1 ON oi1.product_id = p1.product_id
JOIN products p2 ON oi2.product_id = p2.product_id
GROUP BY p1.product_name, p2.product_name
ORDER BY times_bought_together DESC
LIMIT 20;
"""
run_query(q16)

,product_a,product_b,times_bought_together
0,Address Kids,Live Decor,2
1,Agency Footwear,Want Men,2
2,Away Kitchen,Space Women,2
3,Blood Mobiles,Space Women,2
4,Boy Cameras,Want Men,2
5,Everyone Kitchen,May Accessories,2
6,Fact Bedding,Thank Non-Fiction,2
7,Himself Laptops,While Accessories,2
8,Indeed Comics,Mean Non-Fiction,2
9,Item Accessories,Seek Mobiles,2


## Step 7: Customer Segmentation (Frequency, Spend Tier, AOV, RFM) & Churn (Step 6)

In [18]:
# Query 17 (Step 7): Segment customers by purchase frequency
q17 = """
WITH order_counts AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS order_count
    FROM orders
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id
)
SELECT
    customer_id,
    order_count,
    CASE
        WHEN order_count = 1 THEN 'One-time'
        WHEN order_count BETWEEN 2 AND 4 THEN 'Occasional'
        ELSE 'Loyal'
    END AS frequency_segment
FROM order_counts
ORDER BY order_count DESC;
"""
run_query(q17)

,customer_id,order_count,frequency_segment
0,80,6,Loyal
1,287,6,Loyal
2,17,5,Loyal
3,146,5,Loyal
4,212,5,Loyal
...,...,...,...
383,488,1,One-time
384,491,1,One-time
385,495,1,One-time
386,499,1,One-time


In [19]:
# Query 18 (Step 7): Segment customers by spend tier
q18 = """
WITH customer_spend AS (
    SELECT o.customer_id,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS total_spend
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.customer_id
)
SELECT
    customer_id,
    ROUND(total_spend, 2) AS total_spend,
    CASE
        WHEN total_spend < 5000 THEN 'Low'
        WHEN total_spend BETWEEN 5000 AND 20000 THEN 'Medium'
        ELSE 'High'
    END AS spend_tier
FROM customer_spend
ORDER BY total_spend DESC;
"""
run_query(q18)

,customer_id,total_spend,spend_tier
0,306,198393.35,High
1,41,180004.80,High
2,287,141972.61,High
3,383,139595.70,High
4,316,138630.01,High
...,...,...,...
366,173,-14193.21,Low
367,208,-18084.35,Low
368,25,-21295.64,Low
369,429,-27701.54,Low


In [20]:
# Query 19 (Step 4/7): Average Order Value (AOV) by frequency segment
q19 = """
WITH order_value AS (
    SELECT o.order_id, o.customer_id,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS order_value
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL
    GROUP BY o.order_id, o.customer_id
),
order_counts AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS order_count
    FROM orders WHERE customer_id IS NOT NULL GROUP BY customer_id
),
segmented AS (
    SELECT ov.customer_id, ov.order_value,
        CASE WHEN oc.order_count = 1 THEN 'One-time'
             WHEN oc.order_count BETWEEN 2 AND 4 THEN 'Occasional'
             ELSE 'Loyal' END AS frequency_segment
    FROM order_value ov
    JOIN order_counts oc ON ov.customer_id = oc.customer_id
)
SELECT frequency_segment, COUNT(*) AS num_orders, ROUND(AVG(order_value), 2) AS avg_order_value
FROM segmented
GROUP BY frequency_segment
ORDER BY avg_order_value DESC;
"""
run_query(q19)

,frequency_segment,num_orders,avg_order_value
0,Occasional,477,21816.06
1,Loyal,62,19783.43
2,One-time,147,19680.93


In [21]:
# Query 20 (Step 7): RFM Analysis (Recency, Frequency, Monetary)
q20 = """
WITH last_order AS (
    SELECT customer_id, MAX(order_date) AS last_order_date, COUNT(DISTINCT order_id) AS frequency
    FROM orders WHERE customer_id IS NOT NULL GROUP BY customer_id
),
monetary AS (
    SELECT o.customer_id, SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent / 100.0)) AS monetary
    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.customer_id IS NOT NULL GROUP BY o.customer_id
),
rfm_base AS (
    SELECT lo.customer_id,
        CAST(JULIANDAY((SELECT MAX(order_date) FROM orders)) - JULIANDAY(lo.last_order_date) AS INTEGER) AS recency_days,
        lo.frequency, m.monetary
    FROM last_order lo JOIN monetary m ON lo.customer_id = m.customer_id
),
rfm_scored AS (
    SELECT customer_id, recency_days, frequency, ROUND(monetary, 2) AS monetary,
        NTILE(5) OVER (ORDER BY recency_days ASC) AS r_score,
        NTILE(5) OVER (ORDER BY frequency DESC) AS f_score,
        NTILE(5) OVER (ORDER BY monetary DESC) AS m_score
    FROM rfm_base
)
SELECT customer_id, recency_days, frequency, monetary, r_score, f_score, m_score,
    (r_score + f_score + m_score) AS rfm_total,
    CASE
        WHEN (r_score + f_score + m_score) >= 12 THEN 'Champion'
        WHEN (r_score + f_score + m_score) >= 9  THEN 'Loyal'
        WHEN (r_score + f_score + m_score) >= 6  THEN 'At Risk'
        ELSE 'Lost'
    END AS rfm_segment
FROM rfm_scored
ORDER BY rfm_total DESC;
"""
run_query(q20)

,customer_id,recency_days,frequency,monetary,r_score,f_score,m_score,rfm_total,rfm_segment
0,239,239,1,664.81,5,5,5,15,Champion
1,469,245,1,21.77,5,5,5,15,Champion
2,85,256,1,8994.51,5,5,5,15,Champion
3,229,257,1,6513.46,5,5,5,15,Champion
4,31,257,1,1548.67,5,5,5,15,Champion
...,...,...,...,...,...,...,...,...,...
366,383,31,3,139595.70,1,1,1,3,Lost
367,412,32,3,127430.31,1,1,1,3,Lost
368,306,33,5,198393.35,1,1,1,3,Lost
369,7,33,3,75698.00,1,1,1,3,Lost


In [22]:
# Query 21 (Step 6): Churned vs Repeat customers
q21 = """
WITH customer_orders AS (
    SELECT customer_id, COUNT(DISTINCT order_id) AS order_count, MAX(order_date) AS last_order_date
    FROM orders WHERE customer_id IS NOT NULL GROUP BY customer_id
),
dataset_max_date AS (SELECT MAX(order_date) AS max_date FROM orders)
SELECT co.customer_id, co.order_count, co.last_order_date,
    CAST(JULIANDAY(d.max_date) - JULIANDAY(co.last_order_date) AS INTEGER) AS days_since_last_order,
    CASE WHEN JULIANDAY(d.max_date) - JULIANDAY(co.last_order_date) > 60 THEN 'Churned' ELSE 'Active' END AS churn_status,
    CASE WHEN co.order_count > 1 THEN 'Repeat' ELSE 'One-time' END AS repeat_status
FROM customer_orders co, dataset_max_date d
ORDER BY days_since_last_order DESC;
"""
run_query(q21)

,customer_id,order_count,last_order_date,days_since_last_order,churn_status,repeat_status
0,322,1,2025-08-05 22:59:27,363,Churned,One-time
1,366,1,2025-08-08 15:54:17,360,Churned,One-time
2,243,1,2025-08-19 14:12:06,349,Churned,One-time
3,155,1,2025-08-26 00:00:00,343,Churned,One-time
4,77,1,2025-08-27 19:54:08,341,Churned,One-time
...,...,...,...,...,...,...
383,377,1,2026-08-02 00:54:22,2,Active,One-time
384,320,3,2026-08-02 19:50:04,1,Active,Repeat
385,149,2,2026-08-03 19:26:41,0,Active,Repeat
386,303,3,2026-08-04 13:34:34,0,Active,Repeat
